In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Sun Jul 19 21:45:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import os
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file

# Configurations

In [5]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'vi'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
LORA_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260719113226'
LORA_LANG_CKPT_DIR = 'checkpoint-120'
LORA_LANG_CKPT_STEP = 120
LORA_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-v260711104723'
LORA_TASK_CKPT_DIR = 'checkpoint-320'
LORA_TASK_CKPT_STEP = 320
ADDITION_TYPE = 'Addition'

# Set up the addition weights
addition_weights = [0.5, 0.5] if ADDITION_TYPE == 'Averaging' else [1.0, 1.0]

# Set up the hub merged model ID
assert SRC_LANG in LORA_TASK_ID and TGT_LANG in LORA_LANG_ID, "LoRA IDs do not match the specified source language and target language."
model_id_prefix, model_id_suffix = LORA_TASK_ID.split(SRC_LANG)
data_size_str = model_id_suffix.split('LoRA')[0].replace('-', '')
hub_merged_model_id = f"{model_id_prefix}{TGT_LANG}-{data_size_str}-LoRA-{ADDITION_TYPE}-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719214552


# Utilities

In [6]:
# LoRA utilities
def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

# Model

In [7]:
# Download the language and task LoRA adapters
_, lora_lang_dir = download_hf_model(repo_id=LORA_LANG_ID, ckpt_step=LORA_LANG_CKPT_STEP)
_, lora_task_dir = download_hf_model(repo_id=LORA_TASK_ID, ckpt_step=LORA_TASK_CKPT_STEP)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 101 files:   0%|          | 0/101 [00:00<?, ?it/s]

Fetching 197 files:   0%|          | 0/197 [00:00<?, ?it/s]

In [8]:
# Create directories for fixed LoRA adapters
lora_lang_fixed_dir = f'{lora_lang_dir}_fixed'
lora_task_fixed_dir = f'{lora_task_dir}_fixed'

!mkdir -p $lora_lang_fixed_dir
!mkdir -p $lora_task_fixed_dir
!cp -r $lora_lang_dir/* $lora_lang_fixed_dir
!cp -r $lora_task_dir/* $lora_task_fixed_dir
!rm $lora_lang_fixed_dir/adapter_model.safetensors
!rm $lora_task_fixed_dir/adapter_model.safetensors

In [9]:
# Load the LoRA state dicts and fix them
lora_lang_state_dict = load_file(f'{lora_lang_dir}/adapter_model.safetensors')
lora_task_state_dict = load_file(f'{lora_task_dir}/adapter_model.safetensors')

lora_lang_fixed_state_dict = {k: v for k, v in lora_lang_state_dict.items() if 'lm_head' not in k}
lora_task_fixed_state_dict = lora_task_state_dict

save_file(lora_lang_fixed_state_dict, f'{lora_lang_fixed_dir}/adapter_model.safetensors')
save_file(lora_task_fixed_state_dict, f'{lora_task_fixed_dir}/adapter_model.safetensors')

In [10]:
# Sanity check
for i in range(11):
    print(f"Task LoRA layer-{i} attention value weight norm:", lora_task_fixed_state_dict[f'base_model.model.roberta.encoder.layer.{i}.attention.self.value.lora_A.weight'].norm().item())
print()
print("Task LoRA qa_outputs weight norm:", lora_task_fixed_state_dict['base_model.model.qa_outputs.weight'].norm().item())
print("Task LoRA qa_outputs bias norm:", lora_task_fixed_state_dict['base_model.model.qa_outputs.bias'].norm().item())

Task LoRA layer-0 attention value weight norm: 2.4282357692718506
Task LoRA layer-1 attention value weight norm: 2.398138999938965
Task LoRA layer-2 attention value weight norm: 2.488279104232788
Task LoRA layer-3 attention value weight norm: 2.537689685821533
Task LoRA layer-4 attention value weight norm: 2.451260566711426
Task LoRA layer-5 attention value weight norm: 2.4291555881500244
Task LoRA layer-6 attention value weight norm: 2.4716663360595703
Task LoRA layer-7 attention value weight norm: 2.502314805984497
Task LoRA layer-8 attention value weight norm: 2.4796252250671387
Task LoRA layer-9 attention value weight norm: 2.4592087268829346
Task LoRA layer-10 attention value weight norm: 2.4371721744537354

Task LoRA qa_outputs weight norm: 0.9504339098930359
Task LoRA qa_outputs bias norm: 0.010722421109676361


In [11]:
from peft import PeftConfig

# A trick because peft makes the head of a question-answering model to be two layers: the original and trainable qa_outputs layers
# This seems to make the LoRA merging to not work as expected internally and make the LoRA-merged model to perform poorly
config = PeftConfig.from_pretrained(lora_task_fixed_dir)
config.task_type = 'FEATURE_EXTRACTION'
config.modules_to_save = None
config.save_pretrained(lora_task_fixed_dir)

In [12]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

Base model layer-0 attention value weight mean: -5.437493382487446e-05
Base model layer-1 attention value weight mean: 8.216523565351963e-05
Base model layer-2 attention value weight mean: -4.029457727483532e-07
Base model layer-3 attention value weight mean: -6.332253542495891e-05
Base model layer-4 attention value weight mean: -5.2573545872292016e-06
Base model layer-5 attention value weight mean: 2.8999413189012557e-05
Base model layer-6 attention value weight mean: -0.0002365001564612612
Base model layer-7 attention value weight mean: 2.0596125978045166e-05
Base model layer-8 attention value weight mean: 6.689861038466915e-05
Base model layer-9 attention value weight mean: 3.7930796679574996e-05
Base model layer-10 attention value weight mean: -5.502262865775265e-05

Base model qa_outputs weight mean: -4.435734354046872e-06
Base model qa_outputs bias mean: 0.0


In [14]:
# Load the task and language LoRA adapters into the base model
# lora_model = PeftModel.from_pretrained(base_model, model_id=LORA_TASK_ID, subfolder=LORA_TASK_CKPT_DIR, adapter_name='task')
# lora_model.load_adapter(model_id=LORA_LANG_ID, subfolder=LORA_LANG_CKPT_DIR, adapter_name='lang')
lora_model = PeftModel.from_pretrained(base_model, lora_task_fixed_dir, adapter_name='task')
lora_model.load_adapter(lora_lang_fixed_dir, adapter_name='lang')

# Combine the task and language LoRA adapters into a single LoRA adapter via weighted addition
lora_model.add_weighted_adapter(
    adapters=['task', 'lang'],
    weights=addition_weights,
    combination_type='linear',
    adapter_name='lora_addition'
)
lora_model.set_adapter('lora_addition')

# Merge the combined LoRA adapter into the base model
merged_model = lora_model.merge_and_unload()

In [ ]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

Merged model layer-0 attention value weight mean: -5.430883538792841e-05
Merged model layer-1 attention value weight mean: 8.357434126082808e-05
Merged model layer-2 attention value weight mean: -2.921558916568756e-06
Merged model layer-3 attention value weight mean: -6.286885036388412e-05
Merged model layer-4 attention value weight mean: -5.940667506365571e-06
Merged model layer-5 attention value weight mean: 2.9140941478544846e-05
Merged model layer-6 attention value weight mean: -0.00023704195336904377
Merged model layer-7 attention value weight mean: 2.1249687051749788e-05
Merged model layer-8 attention value weight mean: 6.679810758214444e-05
Merged model layer-9 attention value weight mean: 3.559467222657986e-05
Merged model layer-10 attention value weight mean: -5.601425436907448e-05

Merged model qa_outputs weight mean: -0.0005476757069118321
Merged model qa_outputs bias mean: -0.007573713548481464


In [ ]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...yrmwetb/model.safetensors:   1%|          | 7.98MB / 1.11GB            

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcgaa3l99/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

  ...9/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719214552


: 